# 02 — Site history, storage vs. sequestration, and terrain covariates
**Fernow Experimental Forest · WS4 (REF) / WS2 (DLC) / WS5 (STS)** — run in ArcGIS Pro Notebooks or Jupyter.

This notebook answers four reviewer questions the 2022 snapshot alone cannot:

| § | Question | Data |
|---|---|---|
| 0 | When was each plot actually last harvested? (**decision gate**) | removal records + 100% census |
| 1–2 | Were the watersheds comparable *before* management began? | 1958–59 100% census |
| 3 | How do **storage** (stock) and **sequestration** (rate) differ among treatments, 1958→2022? | census 1958–2020 + 2022 plots |
| 4 | Do our six 2022 plots per watershed represent their watershed? | census + historic USFS growth plots |
| 5 | Does terrain (elevation, slope, aspect, TPI, TWI, solar, streams) explain AGB-C instead of management? | 5-m DEM derivatives, SSURGO, lidar |
| 6 | Soil × terrain (runs only once the soil-plot → tree-plot key is entered) | soil + terrain |

Shared statistics live in `fernow_stats.py` (same folder as this notebook). Plot = unit of replication;
one watershed per treatment, so inference is conditional on these watersheds (Hurlbert 1984; Oksanen 2001).


In [ ]:
# ============================== CONFIG — edit this cell only ==============================
import os, sys
NOTEBOOK_DIR = os.getcwd()            # folder holding this notebook + fernow_stats.py
DATA_ROOT = r'.'                      # repo root: *_basal_area*.xlsx, 'FEF Timber Data WS2_WS5.xlsx', Cifelli_MS_Thesis.docx
GDB = os.path.join(DATA_ROOT, 'gdb_export')

SAMPLING_DATE = 2022.54               # July 2022 as a decimal year

# Last-harvest year per subcompartment. DEFAULT = written removal records ('FEF Timber Data WS2_WS5.xlsx').
# The 100% census files date WS5A's harvest 2018 (residual tallied 2019) and WS5B's 2012 -> winter entries.
# Edit here if the Fernow data manager confirms different dates; everything downstream updates.
LAST_HARVEST = {'WS2A': 2020, 'WS2B': 2017, 'WS5A': 2019, 'WS5B': 2013}

# Allometry used for the 2022 plots in this notebook: 'published' = Jenkins et al. (2003) Table 4 coefficients
# (identical to what is applied to the historic census, so the time series is internally consistent);
# 'trees_csv' = the agb_kg_jenkins column (hard-maple group runs 3-5% below published; source unknown).
AGB_SOURCE = 'published'

# Soil plot -> tree plot key (needed only for Section 6). Fill from field notes, e.g. {7: 'WS2A-1', ...}.
SOIL_TREE_KEY = {}

N_PERM = 9999                          # permutations for Freedman-Lane tests
# ==========================================================================================
sys.path.insert(0, NOTEBOOK_DIR)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
from fernow_stats import *
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)
for f in ['ws2_basal_area_mod.xlsx', 'ws4_basal_area.xlsx', 'ws5_basal_area.xlsx', 'FEF Timber Data WS2_WS5.xlsx']:
    assert os.path.exists(os.path.join(DATA_ROOT, f)), f'missing {f} — check DATA_ROOT'
assert os.path.exists(os.path.join(GDB, 'trees.csv')), 'missing gdb_export — check DATA_ROOT'
print('OK — data found')

## §0 — DECISION GATE: harvest dates from the written records

The manuscript/thesis and the Phase-1 handoff state *"WS5 last harvested 2013."* The removal records disagree
for the subcompartment where most WS5 plots sit. Per the project rule (**records win over recollection**), the
table below is built from the records themselves, then each 2022 plot is assigned its subcompartment's
last-harvest year using the *mapped* subcompartment polygon (`plots_study.csv: subcomp_mapped`).


In [ ]:
timber = pd.read_excel(os.path.join(DATA_ROOT, 'FEF Timber Data WS2_WS5.xlsx'))
rec = (timber.groupby(['Compartment', 'Year of Harvest'])['Number of Stems'].sum()
       .reset_index().rename(columns={'Number of Stems': 'stems_removed_records'}))
print('Written removal records (FEF Timber Data):'); print(rec.to_string(index=False))

plots = pd.read_csv(os.path.join(GDB, 'plots_study.csv'))
plots['last_harvest'] = plots['subcomp_mapped'].map(LAST_HARVEST)
plots['yrs_since_harvest'] = (SAMPLING_DATE - plots['last_harvest']).round(1)
print('\nTree plots, mapped subcompartment, and time since last harvest at July-2022 sampling:')
print(plots[['plot_id', 'watershed', 'usfs_subcomp', 'subcomp_mapped', 'cutting_cycle_mapped',
             'site_index_mapped', 'last_harvest', 'yrs_since_harvest']].to_string(index=False))
print('\nMedian years since harvest by watershed:',
      plots.groupby('watershed')['yrs_since_harvest'].median().to_dict())

**What this changes.** With the recorded dates, 5 of 6 plots in each managed watershed were **2 yr (DLC, WS2A)
and ~3.5 yr (STS, WS5A)** post-harvest; one plot each sits in the longer-rotation B subcompartment (5 yr / ~9.5 yr).
The "perfect 2-vs-9-year confound" in the handoff came from the unverified 2013 date. DLC vs. STS is therefore
primarily a **cutting-intensity** contrast (large-tree removal at 15-yr intervals vs. lighter removal at 10-yr
intervals, quantified in §3), not a recovery-time contrast. The manuscript's Methods date for WS5 must be corrected.

Label notes for the record: plots `WS2B-1` and `WS2B-2` carry a B label but fall inside the WS2A polygon (the thesis
states only *one* DLC plot came from WS2B, consistent with the map). In the soil file, WS5 plots are labelled
5×`WS5b` + 1×`WS5a` — the inverse of the thesis text and the tree-plot map (handled in notebook 03).


## §1 — Reconstructing aboveground carbon from the 1958–2020 100% census

Each `wsXx stand >5in` sheet is a 100% tally of stems ≥5 in (12.7 cm) DBH by species (FEF code) × 2-in DBH class,
for every census (`initial`, `recruise`) and every harvest (`harvest` = stems removed, `residual` = recruise − cut).
AGB is computed per stem at the class value with the **same Jenkins et al. (2003) equations** used for the 2022 plots,
× 0.48 for carbon, and expanded by subcompartment area from each file's metadata sheet. Using class values rather than
exact diameters introduces a small Jensen's-inequality bias (slight underestimate) that is identical across treatments.


In [ ]:
def read_areas(fn):
    m = pd.read_excel(fn, sheet_name='metadata', header=None)
    stop = m.index[m[0].astype(str).str.contains('GROWTH', na=False)][0]
    out = {}
    for _, r in m.iloc[:stop].iterrows():
        if str(r[0]).startswith('WS ') and pd.notna(r[1]):
            try: out[str(r[0]).replace('WS ', 'WS').upper()] = float(r[1])
            except ValueError: pass
    return out

census, areas = [], {}
for fn in ['ws2_basal_area_mod.xlsx', 'ws4_basal_area.xlsx', 'ws5_basal_area.xlsx']:
    path = os.path.join(DATA_ROOT, fn); areas.update(read_areas(path))
    for sh in [s for s in pd.ExcelFile(path).sheet_names if 'stand >5in' in s]:
        d = pd.read_excel(path, sheet_name=sh); d.columns = [c.strip() for c in d.columns]
        d = d[['year', 'measurement', 'type', 'spp', '2-in dbh class', 'total stems']].copy()
        d.columns = ['year', 'measurement', 'type', 'spp', 'dbh_in', 'stems']
        d['subcomp'] = sh.split()[0].upper()          # e.g. 'WS2A'
        census.append(d)
census = pd.concat(census, ignore_index=True)
census['watershed'] = census['subcomp'].str[:3]
census['jenkins_group'] = census['spp'].map(FEF_TO_JENKINS).fillna('mixed hardwood')
census['unmapped_spp'] = ~census['spp'].isin(FEF_TO_JENKINS.keys())
census['dbh_cm'] = census['dbh_in'] * IN_TO_CM
census['agb_kg_stem'] = [jenkins_agb_kg(d, g) for d, g in zip(census['dbh_cm'], census['jenkins_group'])]
census['C_kg'] = census['agb_kg_stem'] * census['stems'] * C_FRACTION
census['BA_ft2'] = np.pi * (census['dbh_in'] / 24) ** 2 * census['stems']
census['area_ha'] = census['subcomp'].map(areas) * AC_TO_HA

print('Subcompartment areas (ac):', areas)
print('Measurement types:', sorted(census.measurement.unique()))
un = census.loc[census.unmapped_spp, 'BA_ft2'].sum() / census['BA_ft2'].sum() * 100
print(f'Basal area in FEF species codes without a verified name (assigned mixed hardwood): {un:.1f}%')

agg = (census.groupby(['watershed', 'subcomp', 'year', 'measurement'])
       .agg(C_kg=('C_kg', 'sum'), BA_ft2=('BA_ft2', 'sum'), stems=('stems', 'sum'), area_ha=('area_ha', 'first'))
       .reset_index())
agg['C_MgHa'] = agg['C_kg'] / 1000 / agg['area_ha']
agg['BA_m2Ha'] = agg['BA_ft2'] * 0.092903 / agg['area_ha']
agg['stems_ha'] = agg['stems'] / agg['area_ha']
agg.round(2).head(12)

## §2 — Pre-treatment baseline (1958–59): were the watersheds equivalent before management?

This is the strongest available defence against site confounding (§5 shows REF plots sit higher and on gentler
slopes). If stocking was equivalent **before** treatments began, the 2022 differences arose after management started —
a before/after-control logic that a plot-level ANCOVA with n = 18 cannot provide.


In [ ]:
base = agg[agg.measurement == 'initial'].copy()
base['treatment'] = base['watershed'].map(LABELS)
base['QMD_cm'] = np.sqrt(base['BA_m2Ha'] / base['stems_ha'] / np.pi) * 200
print('Pre-treatment (initial) census, stems >=12.7 cm:')
print(base[['subcomp', 'treatment', 'year', 'C_MgHa', 'BA_m2Ha', 'stems_ha', 'QMD_cm']].round(1).to_string(index=False))

ref_b = base.loc[base.watershed == 'WS4', 'C_MgHa'].values
man_b = base.loc[base.watershed != 'WS4', 'C_MgHa'].values
print(f'\nBaseline AGB-C: REF subcompartments {ref_b.min():.0f}-{ref_b.max():.0f}, managed {man_b.min():.0f}-{man_b.max():.0f} Mg C/ha')
print(f'Difference in means (managed - REF): {man_b.mean() - ref_b.mean():+.1f} Mg C/ha '
      f'({(man_b.mean() / ref_b.mean() - 1) * 100:+.0f}%); Welch t p = {stats.ttest_ind(man_b, ref_b, equal_var=False).pvalue:.2f} '
      f'(n = {len(man_b)} vs {len(ref_b)} subcompartments — descriptive)')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
b = base.sort_values(['watershed', 'subcomp'])
cols = [COLORS[w] for w in b.watershed]
ax[0].bar(b.subcomp, b.C_MgHa, color=cols); ax[0].set_ylabel('AGB-C, stems ≥12.7 cm (Mg ha$^{-1}$)')
ax[0].set_title('Pre-treatment census (1958–59)')
ax[1].bar(b.subcomp, b.BA_m2Ha, color=cols); ax[1].set_ylabel('Basal area (m$^2$ ha$^{-1}$)')
ax[1].set_title('Pre-treatment basal area')
for a in ax: a.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

## §3 — Storage vs. sequestration, 1958 → 2022

Definitions used throughout the manuscript (standard forest-inventory accounting):

- **Storage (stock)** — aboveground C standing at a point in time (Mg C ha⁻¹).
- **Net growth** — the rate at which the *standing* forest adds C: survivor growth + ingrowth − mortality, between
  censuses with no cutting in between (Mg C ha⁻¹ yr⁻¹). This is the "sequestration" rate of the live trees.
- **Removals** — C exported in harvests (leaves the site; part enters wood products, not tracked here).
- **Net on-site change** — change in stock including removals = net growth − removals. This is what the atmosphere
  "sees" from the forest itself.

A managed stand can have *higher net growth* yet *lower storage* and *~zero net on-site change* — the
sawtooth. That is the storage/sequestration distinction in one figure.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
styles = {'WS4A': '-', 'WS4B': '--', 'WS4C': ':', 'WS2A': '-', 'WS2B': '--', 'WS5A': '-', 'WS5B': '--'}
order_meas = {'initial': 0, 'recruise': 1, 'harvest': 2, 'residual': 3}
for sc, d in agg[agg.measurement.isin(['initial', 'recruise', 'residual'])].groupby('subcomp'):
    d = d.assign(o=d.measurement.map(order_meas)).sort_values(['year', 'o'])
    ax.plot(d.year, d.C_MgHa, styles[sc], color=COLORS[sc[:3]], marker='o', ms=3, lw=1.4, label=sc)

# 2022 plots (stems >=12.7 cm, same allometry) as mean +- SE per watershed
trees = pd.read_csv(os.path.join(GDB, 'trees.csv')); trees = trees[trees.dbh_cm.notna()].copy()
trees['agb_pub'] = [jenkins_agb_kg(d, g) for d, g in zip(trees.dbh_cm, trees.jenkins_group)]
trees['agb_use'] = trees['agb_pub'] if AGB_SOURCE == 'published' else trees['agb_kg_jenkins']
PLOT_HA = 404.70782 / 1e4
p5 = (trees[trees.dbh_in >= 5].groupby(['watershed', 'plot_id'])['agb_use'].sum() * C_FRACTION / 1000 / PLOT_HA).rename('C5').reset_index()
for w in ORDER:
    v = p5[p5.watershed == w].C5
    ax.errorbar(SAMPLING_DATE + {'WS4': -0.4, 'WS2': 0, 'WS5': 0.4}[w], v.mean(), yerr=v.std(ddof=1) / np.sqrt(len(v)),
                fmt='D', color=COLORS[w], ms=7, mec='k', capsize=3)
ax.text(2020.3, ax.get_ylim()[1] * 0.97, '◆ 2022 plots\n(mean ± SE)', fontsize=8, va='top')
ax.set_xlabel('Year'); ax.set_ylabel('AGB-C, stems ≥12.7 cm (Mg ha$^{-1}$)')
ax.set_title('Aboveground C storage by subcompartment, 100% census 1958–2020')
ax.legend(ncol=4, fontsize=8, loc='lower left'); plt.tight_layout(); plt.show()

In [ ]:
# Carbon accounting per subcompartment over its full record, then area-weighted to watershed
rows = []
for sc, d in agg.groupby('subcomp'):
    ini = d[d.measurement == 'initial'].iloc[0]
    live = d[d.measurement.isin(['recruise', 'residual'])]
    last = live.assign(o=live.measurement.map(order_meas)).sort_values(['year', 'o']).iloc[-1]
    removed = d.loc[d.measurement == 'harvest', 'C_MgHa'].sum()
    n_entries = d.loc[d.measurement == 'harvest', 'year'].nunique()
    yrs = last.year - ini.year
    rows.append(dict(subcomp=sc, watershed=sc[:3], area_ha=ini.area_ha, start=int(ini.year), end=int(last.year),
                     C_start=ini.C_MgHa, C_end=last.C_MgHa, removed=removed, entries=n_entries,
                     net_onsite_rate=(last.C_MgHa - ini.C_MgHa) / yrs,
                     net_growth_rate=(last.C_MgHa - ini.C_MgHa + removed) / yrs,
                     removal_rate=removed / yrs))
acct = pd.DataFrame(rows)
print('Per-subcompartment accounting (Mg C/ha and Mg C/ha/yr):')
print(acct.round(2).to_string(index=False))

wmean = lambda g, c: np.average(g[c], weights=g['area_ha'])
ws_acct = pd.DataFrame({c: acct.groupby('watershed').apply(lambda g: wmean(g, c))
                        for c in ['C_start', 'C_end', 'removed', 'net_growth_rate', 'removal_rate', 'net_onsite_rate']}).reindex(ORDER)
ws_acct.index = [LABELS[w] for w in ws_acct.index]
print('\nArea-weighted by watershed:'); ws_acct.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(3)
ax.bar(x - 0.2, ws_acct['net_growth_rate'], 0.4, color='#6a994e', label='Net growth (live-tree uptake)')
ax.bar(x + 0.2, ws_acct['net_onsite_rate'], 0.4, color='#bc4749', label='Net on-site change (growth − removals)')
ax.axhline(0, color='k', lw=0.8); ax.set_xticks(x); ax.set_xticklabels(ws_acct.index)
ax.set_ylabel('Mg C ha$^{-1}$ yr$^{-1}$ (stems ≥12.7 cm)')
ax.set_title('Sequestration rate vs. change in storage, 1958–2020'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Period-level growth vs. starting stock.** Every between-census interval without cutting is a natural experiment
in how fast C accumulates at a given stocking. Managed stands spend their time at low–moderate stock where growth is
fast; REF sits at high stock where growth is slower but positive. These points are overlaid on the West Virginia
FIA remeasurement curve in notebook 04.


In [ ]:
per = []
for sc, d in agg[agg.measurement.isin(['initial', 'recruise', 'residual'])].groupby('subcomp'):
    d = d.assign(o=d.measurement.map(order_meas)).sort_values(['year', 'o']).reset_index(drop=True)
    for i in range(len(d) - 1):
        a, b = d.iloc[i], d.iloc[i + 1]
        if a.measurement in ('initial', 'recruise', 'residual') and b.measurement == 'recruise' and b.year > a.year:
            per.append(dict(subcomp=sc, watershed=sc[:3], y0=a.year, y1=b.year, C0=a.C_MgHa, BA0_ft2ac=a.BA_m2Ha / FT2AC_TO_M2HA,
                            rate_C=(b.C_MgHa - a.C_MgHa) / (b.year - a.year),
                            rate_BA_ft2ac=(b.BA_m2Ha - a.BA_m2Ha) / FT2AC_TO_M2HA / (b.year - a.year)))
periods = pd.DataFrame(per)
periods.to_csv(os.path.join(NOTEBOOK_DIR, 'fernow_census_periods.csv'), index=False)   # used by notebook 04
r = stats.spearmanr(periods.C0, periods.rate_C)
print(f'{len(periods)} uncut census intervals. Spearman(start stock, net growth rate) rho = {r.correlation:.2f}, p = {r.pvalue:.4f}')
fig, ax = plt.subplots(figsize=(5.5, 4))
for w in ORDER:
    d = periods[periods.watershed == w]; ax.scatter(d.C0, d.rate_C, color=COLORS[w], label=LABELS[w], s=35, edgecolor='k')
ax.axhline(0, color='k', lw=0.6); ax.set_xlabel('Stock at start of interval (Mg C ha$^{-1}$)')
ax.set_ylabel('Net growth (Mg C ha$^{-1}$ yr$^{-1}$)'); ax.legend(); ax.set_title('Fernow census intervals without cutting')
plt.tight_layout(); plt.show()
periods.round(2)

In [ ]:
# Harvest intensity per entry (for the DLC-vs-STS intensity framing)
h = agg[agg.measurement == 'harvest'][['subcomp', 'year', 'C_MgHa', 'BA_m2Ha']].rename(columns={'C_MgHa': 'C_removed', 'BA_m2Ha': 'BA_removed'})
pre = agg[agg.measurement.isin(['initial', 'recruise'])][['subcomp', 'year', 'C_MgHa']]
h = pd.merge_asof(h.sort_values('year'), pre.sort_values('year').rename(columns={'C_MgHa': 'C_pre', 'year': 'pre_year'}),
                  left_on='year', right_on='pre_year', by='subcomp', direction='backward', tolerance=3)
h['pct_C_removed'] = h['C_removed'] / h['C_pre'] * 100
print('Per-entry removal (pre-harvest census within 3 yr; NaN where none):')
print(h.sort_values(['subcomp', 'year']).round(1).to_string(index=False))
print('\nMedian % of AGB-C removed per entry:', h.groupby(h.subcomp.str[:3]).pct_C_removed.median().round(1).rename(LABELS).to_dict())

## §4 — Do the six 2022 plots represent their watershed?

Compare 2022 plot means (stems ≥12.7 cm, same allometry) to the most recent 100% census, and to the historic 0.5–0.6-ac
USFS growth plots. Two outcomes matter: (i) REF agreement is also an **empirical test of the 0.1-ac plot size**
(if plots were 0.5 ac, REF basal area would be ~5× too low), and (ii) any systematic plot-vs-census gap in managed
watersheds tells us whether plot-based treatment differences are conservative or inflated.


In [ ]:
p5ba = (trees[trees.dbh_in >= 5].groupby(['watershed', 'plot_id'])['ba_m2'].sum() / PLOT_HA).rename('BA').reset_index()
latest = (agg[agg.measurement.isin(['recruise', 'residual'])]
          .assign(o=lambda d: d.measurement.map(order_meas)).sort_values(['year', 'o']).groupby('subcomp').tail(1))
lw = latest.groupby('watershed').apply(lambda g: pd.Series({
    'census_year': f"{int(g.year.min())}-{int(g.year.max())}",
    'census_BA_m2ha': np.average(g.BA_m2Ha, weights=g.area_ha),
    'census_C_MgHa': np.average(g.C_MgHa, weights=g.area_ha)}))
comp = pd.DataFrame({'plot2022_BA_m2ha': p5ba.groupby('watershed').BA.mean(),
                     'plot2022_BA_if_0.5ac': p5ba.groupby('watershed').BA.mean() * 0.1 / 0.5,
                     'plot2022_C_MgHa': p5.groupby('watershed').C5.mean()}).join(lw).reindex(ORDER)
# project each subcompartment's latest census to July 2022 at its own long-run net growth rate (acct, §3)
latest = latest.merge(acct[['subcomp', 'net_growth_rate']], on='subcomp')
latest['C_proj2022'] = latest.C_MgHa + latest.net_growth_rate * (SAMPLING_DATE - latest.year)
comp = comp.join(latest.groupby('watershed').apply(lambda g: np.average(g.C_proj2022, weights=g.area_ha)).rename('census_C_proj2022'))
comp['plot/census_proj'] = comp['plot2022_C_MgHa'] / comp['census_C_proj2022']
comp.index = [LABELS[w] for w in comp.index]
comp.round(2)

In [ ]:
# Historic USFS growth plots (main plots), stems >=5 in, BA per acre = sum BA / (n plots x plot size)
GP_SIZE = {'WS2A': 0.6, 'WS2B': 0.6, 'WS4A': 0.5, 'WS4B': 0.5, 'WS4C': 0.5, 'WS5A': 0.5, 'WS5B': 0.5}  # ac, metadata sheets
BA5 = np.pi * (5 / 24) ** 2
gp = []
for fn in ['ws2_basal_area_mod.xlsx', 'ws4_basal_area.xlsx', 'ws5_basal_area.xlsx']:
    path = os.path.join(DATA_ROOT, fn)
    for sh in [s for s in pd.ExcelFile(path).sheet_names if s.endswith(' plot')]:
        d = pd.read_excel(path, sheet_name=sh); d.columns = [c.strip() for c in d.columns]
        d = d[(d['plot type'] == 'main') & (d['per tree basal area'] >= BA5)]
        sc = sh.split()[0].upper()
        g = d.groupby('year').agg(ba=('per tree basal area', 'sum'), nplots=('plot', 'nunique')).reset_index()
        g['subcomp'] = sc; g['BA_m2ha'] = g.ba / (g.nplots * GP_SIZE[sc]) * FT2AC_TO_M2HA
        gp.append(g)
gp = pd.concat(gp)
print('Historic growth-plot basal area, stems >=12.7 cm (m2/ha) — compare with census at similar dates:')
print(gp.pivot_table(index='year', columns='subcomp', values='BA_m2ha').round(1).to_string())

**How to read §4.** Use `plot/census_proj` (census projected to 2022 at each subcompartment's own growth rate; REF's
last census is 2001). If the REF ratio is ≈1 while managed ratios are >1, the long-term plot network in
managed watersheds sits in better-stocked patches (plots avoid landings/skid trails). That would make the plot-based
REF − managed gap **conservative** — the true watershed-scale difference would be larger. State this in Methods.


## §5 — Terrain and site covariates

**5a. Covariate balance.** If a covariate differs among watersheds, it is confounded with treatment at n = 1
watershed/treatment. Covariates from `plot_terrain.csv` (5-m DEM, plot-footprint means: elevation, slope [deg],
northness = cos(aspect), eastness = sin(aspect), TPI at 50/200 m, TWI), `plot_solar.csv`, `plot_site.csv`
(stream distance), `plots_study.csv` (mapped site index), `ssurgo_plot.csv` (0–20 cm clay).


In [ ]:
T = pd.read_csv(os.path.join(GDB, 'plot_terrain.csv'))
SO = pd.read_csv(os.path.join(GDB, 'plot_solar.csv'))
SI = pd.read_csv(os.path.join(GDB, 'plot_site.csv'))[['plot_id', 'stream_dist_any_m', 'nrcs_SoilSeries']]
SS = pd.read_csv(os.path.join(GDB, 'ssurgo_plot.csv'))[['plot_id', 'claytotal_r_0_20']]
CA = pd.read_csv(os.path.join(GDB, 'plot_canopy.csv'))
agbC = (trees.groupby(['watershed', 'plot_id'])['agb_use'].sum() * C_FRACTION / 1000 / PLOT_HA).rename('AGBC').reset_index()
site = (plots[['plot_id', 'watershed', 'site_index_mapped', 'yrs_since_harvest']]
        .merge(T, on='plot_id').merge(SO, on='plot_id').merge(SI, on='plot_id').merge(SS, on='plot_id')
        .merge(CA, on='plot_id').merge(agbC, on=['plot_id', 'watershed']))
COVS = {'dem5_fp': 'Elevation (m)', 'slope5_fp': 'Slope (deg)', 'north5_fp': 'Northness', 'east5_fp': 'Eastness',
        'tpi50_fp': 'TPI 50 m', 'tpi200_fp': 'TPI 200 m', 'twi5_fp': 'TWI', 'solar_kwh_m2_yr_fp': 'Solar (kWh/m2/yr)',
        'stream_dist_any_m': 'Stream distance (m)', 'site_index_mapped': 'Site index (ft)', 'claytotal_r_0_20': 'SSURGO clay 0-20 (%)'}
bal = pd.DataFrame([battery_row(lab, [site.loc[site.watershed == w, c].astype(float).values for w in ORDER]) for c, lab in COVS.items()])
print('AGB allometry in use:', AGB_SOURCE, '| plot AGB-C means:',
      site.groupby('watershed').AGBC.mean().reindex(ORDER).round(1).rename(LABELS).to_dict())
bal[['test', 'REF', 'DLC', 'STS', 'ANOVA_p', 'KW_p', 'lead_test']].round(3)

**5b. Does the management effect on AGB-C survive terrain adjustment?** For each covariate, one at a time
(n = 18 cannot support more): `AGB-C ~ watershed + covariate`. Reported: watershed partial-F p (HC3 not needed for
the nested F), a **Freedman–Lane permutation p** (robust to non-normality), adjusted DLC/STS effects vs. REF,
the covariate slope, and ΔAIC vs. the watershed-only model. Collinearity is shown as R² of covariate ~ watershed —
when it is high, the model *cannot* separate the two, and the pre-treatment baseline (§2) is the relevant evidence.


In [ ]:
y = site['AGBC'].values
D = [(site.watershed == 'WS2').astype(float).values, (site.watershed == 'WS5').astype(float).values]
m_ws = ols(y, D, ['DLC', 'STS'])
rows = []
for c, lab in COVS.items():
    x = site[c].astype(float).values
    full = ols(y, D + [x], ['DLC', 'STS', c]); red = ols(y, [x], [c])
    F, pF = nested_F(full, red)
    _, pperm = freedman_lane(y, [x], D, n_perm=N_PERM)
    col = ols(x, D)['r2']
    t = full['table']
    rows.append(dict(covariate=lab, R2_cov_on_ws=col, ws_F=F, ws_p=pF, ws_p_perm=pperm,
                     DLC_adj=t.loc['DLC', 'coef'], STS_adj=t.loc['STS', 'coef'],
                     cov_slope=t.loc[c, 'coef'], cov_p_HC3=t.loc[c, 'p_HC3'], dAIC_vs_wsonly=full['aic'] - m_ws['aic']))
anc = pd.DataFrame(rows)
print(f"Watershed-only model: DLC {m_ws['table'].loc['DLC','coef']:.1f}, STS {m_ws['table'].loc['STS','coef']:.1f} Mg C/ha vs REF; R2 = {m_ws['r2']:.2f}")
anc.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for w in ORDER:
    d = site[site.watershed == w]
    ax[0].scatter(d.dem5_fp, d.AGBC, color=COLORS[w], label=LABELS[w], s=40, edgecolor='k')
    ax[1].scatter(d.slope5_fp, d.AGBC, color=COLORS[w], label=LABELS[w], s=40, edgecolor='k')
ax[0].set_xlabel('Elevation (m)'); ax[1].set_xlabel('Slope (deg)')
for a in ax: a.set_ylabel('AGB-C (Mg ha$^{-1}$)'); a.legend(fontsize=8)
plt.suptitle('Terrain overlap among watersheds — where the covariate ranges do not overlap, ANCOVA extrapolates', fontsize=10)
plt.tight_layout(); plt.show()
print('Elevation range by watershed (m):', site.groupby('watershed').dem5_fp.agg(['min', 'max']).round(0).reindex(ORDER).to_dict('index'))

print('\nWithin-watershed (group-demeaned) correlations with AGB-C, df = n - 3 - 1:')
for c, lab in COVS.items():
    r, p, dfree = within_group_corr(site, c, 'AGBC')
    print(f'  {lab:24s} r = {r:+.2f}  p = {p:.3f}')

**Interpretation guide for §5.** Expect elevation and slope to be imbalanced across watersheds (REF higher and gentler).
If the watershed effect weakens after elevation adjustment, report it plainly: *at the plot scale, topography and
treatment cannot be statistically separated; the pre-treatment census (§2), taken over the full subcompartments at
the same elevations, shows equivalent stocking before management, so the post-treatment divergence is attributed to
management.* A within-watershed *positive* elevation–AGB relationship would be counter to the usual ridge/cove
productivity gradient in Appalachian hardwoods and, at n = 6 per watershed, should be treated as fragile.


**5c. Independent check of field AGB-C with lidar.** Canopy-height metrics (`plot_canopy.csv`) are derived from
airborne lidar entirely independently of the field allometry. Lidar acquisition year is not recorded in the export —
confirm it before citing (a pre-2020 flight would pre-date the latest DLC harvest).


In [ ]:
for c in ['chm_mean_m', 'chm_p50_m', 'chm_p95_m', 'frac_ge20m_leafoff']:
    r = stats.pearsonr(site[c], site.AGBC); rs = stats.spearmanr(site[c], site.AGBC)
    rw, pw, _ = within_group_corr(site, c, 'AGBC')
    print(f'{c:20s} pooled r = {r[0]:+.2f} (p = {r[1]:.4f}); Spearman {rs.correlation:+.2f}; within-watershed r = {rw:+.2f} (p = {pw:.3f})')
print('\nLidar metric by watershed (battery):')
pd.DataFrame([battery_row(c, [site.loc[site.watershed == w, c].values for w in ORDER]) for c in
              ['chm_mean_m', 'chm_p95_m', 'frac_ge20m_leafoff']])[['test', 'REF', 'DLC', 'STS', 'ANOVA_p', 'KW_p', 'lead_test']].round(3)

## §6 — Soil × terrain (gated)

Soil plots 7–24 were sampled at the same growth plots as the trees (thesis Methods), but the export carries no
soil-plot → tree-plot key. Enter `SOIL_TREE_KEY` in the config cell (e.g. `{7: 'WS2A-1', 8: 'WS2A-3', ...}`) and this
section runs ANCOVAs of the main soil responses on the same covariates, plus aboveground–belowground coupling
(AGB-C vs soil C, fine roots vs AGB) at the plot level.


In [ ]:
if len(SOIL_TREE_KEY) != 18:
    print(f'SOIL_TREE_KEY has {len(SOIL_TREE_KEY)}/18 entries — Section 6 skipped. Fill it from field notes to run.')
else:
    soil = pd.read_csv(os.path.join(GDB, 'soil_samples.csv'))
    sp = soil.groupby(['soil_plot', 'depth_code'])[['soil_C_gm2', 'soil_CN', 'fine_root_gm2', 'BD_gcm3']].mean().unstack()
    sp.columns = [f'{a}_{b}' for a, b in sp.columns]
    sp['SoilC_0_40_MgHa'] = (sp['soil_C_gm2_0-20'] + sp['soil_C_gm2_20-40']) / 100
    frac = pd.read_csv(os.path.join(GDB, 'soil_fractions.csv'))
    sp = sp.join(frac[frac.depth_code == '20-40'].set_index('soil_plot')['LF_C_gm2'].rename('LF_C_20_40'))
    sp['plot_id'] = sp.index.map(SOIL_TREE_KEY)
    sd = site.merge(sp.reset_index(), on='plot_id')
    Ds = [(sd.watershed == 'WS2').astype(float).values, (sd.watershed == 'WS5').astype(float).values]
    out = []
    for resp in ['SoilC_0_40_MgHa', 'soil_CN_OH', 'soil_CN_0-20', 'LF_C_20_40', 'BD_gcm3_0-20']:
        ok = sd[resp].notna().values
        for c in ['dem5_fp', 'slope5_fp', 'north5_fp', 'twi5_fp', 'tpi200_fp']:
            yy = sd.loc[ok, resp].values; xx = sd.loc[ok, c].values; DD = [d[ok] for d in Ds]
            full = ols(yy, DD + [xx], ['DLC', 'STS', c]); F, pF = nested_F(full, ols(yy, [xx]))
            out.append(dict(response=resp, covariate=c, n=ok.sum(), ws_p=pF,
                            cov_slope=full['table'].loc[c, 'coef'], cov_p=full['table'].loc[c, 'p_HC3']))
    res6 = pd.DataFrame(out); res6['cov_p_FDR'] = bh_fdr(res6.cov_p)
    print(res6.round(3).to_string(index=False))
    for a, b in [('AGBC', 'SoilC_0_40_MgHa'), ('AGBC', 'fine_root_gm2_0-20'), ('AGBC', 'LF_C_20_40')]:
        r, p, _ = within_group_corr(sd, a, b, method='spearman'); print(f'within-watershed Spearman {a} ~ {b}: rho = {r:+.2f}, p = {p:.3f}')

---
### Outputs carried forward
`fernow_census_periods.csv` (written in §3) is read by notebook 04 to overlay Fernow census growth rates on the
West Virginia FIA remeasurement curve.

### References used in this notebook
Freedman & Lane (1983) *J. Bus. Econ. Stat.* 1:292–298 · Hurlbert (1984) *Ecol. Monogr.* 54:187–211 ·
Jenkins et al. (2003) *For. Sci.* 49:12–35 · Oksanen (2001) *Oikos* 94:27–38 ·
Davis et al. (2009) *For. Ecol. Manage.* 258:2101–2109 (Fernow harvest–C accounting; reconcile against §3) ·
Stephenson et al. (2014) *Nature* 507:90–93 · Luyssaert et al. (2008) *Nature* 455:213–215.
